# 29. How much to settle first, and for how long

Two branches, one per card, in a single session of roughly three to five
hours. Nothing in this notebook needs editing: run it as it is.

Notebook 28 asks whether the narrow end should go first. These two ask
how much of the network that phase should settle, and for how long.

The axis now has five rows. **BF** and **BG** train width 1.00 alone for
10 and 25 epochs and then admit the narrow end: -0.36 and -0.49 against
K, so large-first is answered. **BU** and **BT** are the same
intervention pointed the other way, without and with a freeze at 0.25.
These two move the freeze itself.

**BV settles half the network.** Width 0.50 alone for 25 epochs, then that
block is held still: 2,815,840 weights, 25.1 per cent, against BT's
710,576 at 6.3 per cent. Four times as much. If the narrow-first order
helps at all, this says whether it helps in proportion to how much is
settled before the rest may move, or whether 6.3 per cent was already the
whole effect.

The curriculum width and the freeze width have to match, which is why
this branch also changes what the first phase trains. Freezing 0.50 after
a phase that ran 0.25 alone would lock the channels between them at
their initialisation, and the row would be measuring a partly random
network rather than a settled one.

**BW shortens the phase.** Width 0.25 alone for 10 epochs rather than 25,
then frozen. On the large-first side this knob was worth almost nothing -
BF at 10 epochs reads 73.90 and BG at 25 reads 73.78, a spread of 0.13 -
but nothing there was ever held still. Under a freeze a phase that ends
too early locks a prefix that was not finished, which is a failure mode
the large-first branches cannot have.

## Same counterweight, and a third bug

Thirteen measurements in the literature say the small end is rescued by
weight sharing and the large end pays for it. If that holds here, every
row on this axis is protecting the half that was never in trouble, and
the honest outcome is that all four small-first branches land below K.

And this idea has now found three latent bugs, none of them reachable
before a sampler returned something other than the widest width. Two were
on notebook 28: `chain_target` read before assignment in the real loop,
then the same assumption inside the branch suite's mimic. The third is
BV's: the suite guarded its pair term with "mid_widths is not empty" and
then indexed `mids[0]` and `mids[1]` by hand, which is fine for every
sampler that came before and an IndexError for a phase that settles one
middle width. train.py asks `horizontal_pairs()` and gets an empty list in
that case, so only the mimic was wrong. It now checks for two.

The suite reads 4.5719 for BV and 4.5859 for BW against K's 22.6405, so
it does see both of these.

## `bv_half_first_frozen`

Settle half the network first: width 0.50 alone for 25 epochs, then hold that block still. BT settles the width-0.25 block, which is 710,576 weights - 6.3 per cent. This settles the width-0.50 block instead: 2,815,840 weights, 25.1 per cent, four times as much. If the narrow-first order helps at all, this says whether it helps in proportion to how much of the prefix is settled before the rest is allowed to move, or whether 6.3 per cent was already the whole effect. The curriculum width and the freeze width have to match. Freezing 0.50 after a phase that ran 0.25 alone would lock the channels between them at their initialisation, and the branch would be measuring a partly random network rather than a settled one. Read the five rows of this axis together. Large-first is BF at -0.36 and BG at -0.49 against K; small-first is BU without a freeze, BT with one at 0.25, and these two moving the two knobs a freeze has - how much of the network is settled before the rest may move, and for how long. The same counterweight applies as on notebook 28: thirteen measurements in the literature say the small end is rescued by weight sharing and the large end pays for it, so settling the small end may be protecting the half that was never in trouble.

## `bw_narrow10_frozen`

BT with a shorter first phase: width 0.25 alone for 10 epochs rather than 25, then frozen. This is the same knob BF and BG turn on the other direction, and there it was worth little - 10 epochs of width 1.00 read 73.90 and 25 epochs 73.78, a spread of 0.13. Whether the phase length matters more when the phase is settling a block that then stops moving is the open half of that comparison: under a freeze, a phase that ends too early locks a prefix that was not finished, which has no counterpart in the large-first branches because nothing there was ever held still. Read the five rows of this axis together. Large-first is BF at -0.36 and BG at -0.49 against K; small-first is BU without a freeze, BT with one at 0.25, and these two moving the two knobs a freeze has - how much of the network is settled before the rest may move, and for how long. The same counterweight applies as on notebook 28: thirteen measurements in the literature say the small end is rescued by weight sharing and the large end pays for it, so settling the small end may be protecting the half that was never in trouble.


## Before you start

* Accelerator **GPU T4 x2**, Internet **On**
* Add the CIFAR-100 dataset as an input
* **Save Version -> Save & Run All (Commit)**, not the interactive run.
  An interactive session ends when the browser closes.

The checks at the top run on the CPU and stop the session in about two
minutes if anything is wrong, before a card is touched. Note that the
branch suite mimics the training loop rather than running it, so for
these branches the smoke step below is what actually exercises the new
code. Do not skip it.

If the session times out partway, attach its output to a new copy and
name the logs directory in `RESUME_FROM`. Every epoch writes a
checkpoint, so at most one is lost.

## When it finishes

Send back the final table. Pasting the output of the last cell is enough.

In [ ]:
# Fixed for this notebook. Notebook 29 of 29.
BRANCHES = ['bv_half_first_frozen', 'bw_narrow10_frozen']

SMOKE_FIRST = True

REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'

CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

# To carry a timed-out session forward, attach its output and name the
# logs directory. Leave empty to start fresh.
RESUME_FROM = ''

In [ ]:
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
for i in range(n_gpu):
    print('  {}: {}'.format(i, torch.cuda.get_device_properties(i).name))

WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(CODE):
    subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)

available = sorted(
    name[len('cifar100_'):-len('.yml')]
    for name in os.listdir('apps')
    if name.startswith('cifar100_') and name.endswith('.yml'))
print('\nbranches in apps/:')
for name in available:
    print('   ', name)

missing = [b for b in BRANCHES if b not in available]
if missing:
    raise SystemExit('no config for {}'.format(missing))
if n_gpu < len(BRANCHES):
    print('\n{} branches, {} gpu(s): they will run in sequence.'.format(
        len(BRANCHES), n_gpu))

In [ ]:
# What the chosen branches actually differ in, read off the configs
# rather than from the table above, which can drift.
AXES = ('kd_loss', 'cost_source', 'feature_kd', 'feature_align',
        'feature_layers', 'feature_weight', 'tier_weights',
        'horizontal_kd', 'horizontal_where', 'horizontal_loss',
        'horizontal_weight', 'weight_schedule')

settings = {}
for branch in BRANCHES:
    found = {}
    with open('apps/cifar100_{}.yml'.format(branch)) as handle:
        for line in handle:
            key = line.split(':')[0].strip()
            if key in AXES:
                found[key] = line.split(':', 1)[1].strip()
    settings[branch] = found

print('{:20}'.format('') + ''.join(
    '{:>26}'.format(b) for b in BRANCHES))
for axis in AXES:
    values = [settings[b].get(axis, '-') for b in BRANCHES]
    if all(v == '-' for v in values):
        continue
    print('{:20}'.format(axis) + ''.join(
        '{:>26}'.format(v) for v in values))

In [ ]:
TARGET = 'data/cifar-100-python'
if not os.path.isdir(TARGET):
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        os.symlink(source, TARGET)
        print('linked', source)
    else:
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)
print(sorted(os.listdir(TARGET)))

## Checks, before a card is touched

Seconds on the CPU. Between them these suites have caught a Sinkhorn solved too loosely to have a correct gradient, an alpha-divergence that destroyed the weights in three steps, a calibration that reset the batch norm statistics and never refilled them, and a feature cost normalized so that its own gradient vanished. Every one of those was silent.

The last suite builds each branch in this notebook and runs two training steps of the real loop, profiling included. Four feature branches once reached Kaggle, passed every loss check, and died in the profiler on the first forward, because nothing local had ever called it.

In [ ]:
# Driven by BRANCHES, not by a list written beside it. The branch suite
# used to name its two branches literally, which quietly checked the wrong
# pair for anyone who changed BRANCHES and nothing else.
suites = ['tests/test_loss_ops.py', 'tests/test_bn_calibration.py',
          'tests/test_kd_variants.py',
          'tests/test_channel_reorder.py']
for suite in suites:
    done = subprocess.run([sys.executable, suite], capture_output=True,
                          text=True)
    print('{:34} {}'.format(
        suite, (done.stdout.strip().splitlines() or ['no output'])[-1]))
    if done.returncode != 0:
        print(done.stdout[-3000:], done.stderr[-2000:])
        raise SystemExit('{} failed'.format(suite))

done = subprocess.run(
    [sys.executable, 'tests/test_all_branches.py'] + BRANCHES,
    capture_output=True, text=True)
print()
print(done.stdout.strip()[-2000:])
if done.returncode != 0:
    print(done.stderr[-2000:])
    raise SystemExit('a branch in {} does not build'.format(BRANCHES))

In [ ]:
if RESUME_FROM:
    os.makedirs('logs', exist_ok=True)
    for name in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, name)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join('logs', name),
                            dirs_exist_ok=True)
            print('restored', name)
else:
    print('starting from scratch')

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs, quiet=True):
    """one job per card, output interleaved and tagged"""
    lines = queue.Queue()
    procs = {}
    failing = set()

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {} with {}'.format(
            label, env['CUDA_VISIBLE_DEVICES'], config), flush=True)

    started = time.time()
    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[float(width)] = (
                float(loss), float(top1))
        # Once a branch starts printing a traceback, stop filtering
        # it. The body is indented, so the rule below would keep the
        # exception and throw away where it came from.
        if 'Traceback (most recent call last)' in line:
            failing.add(label)
        if quiet and label not in failing and (
                line.startswith(('  ', ')', 'Model(', 'Total', 'Item'))
                or not line.strip()):
            continue
        print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
if SMOKE_FIRST:
    codes = run_pinned(
        [(b, 'apps/smoke_{}.yml'.format(b)) for b in BRANCHES])
    failed = [b for b, code in codes.items() if code != 0]
    if failed:
        raise SystemExit('smoke failed for {}'.format(failed))
    for b in BRANCHES:
        shutil.rmtree('logs/smoke_{}'.format(b), ignore_errors=True)
    results.clear()
    print('\nsmoke ok')

In [ ]:
codes = run_pinned(
    [(b, 'apps/cifar100_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## Results, against A

A is the number to beat, not C or D. Improving on an ablation of your own
method is not improving on the paper.

One seed, and sigma has not been measured. Three of the four gaps in the
finished table sit between 0.25 and 0.44 points, which is the range where
a single run cannot tell a result from noise.

In [ ]:
# the published run, for reference
A_KL = {0.25: 70.10, 0.30: 70.80, 0.35: 71.50, 0.40: 72.20, 0.45: 72.80,
        0.50: 73.20, 0.55: 73.40, 0.60: 73.80, 0.65: 73.90, 0.70: 74.30,
        0.75: 74.60, 0.80: 74.80, 0.85: 75.10, 0.90: 75.10, 0.95: 75.40,
        1.00: 75.30}

widths = sorted({w for table in results.values() for w in table})
header = '{:>7}{:>9}'.format('width', 'A')
for branch in BRANCHES:
    header += '{:>11}{:>8}'.format(branch[:10], 'vs A')
print(header)

for width in widths:
    row = '{:>7.2f}{:>9.2f}'.format(width, A_KL.get(width, float('nan')))
    for branch in BRANCHES:
        entry = results.get(branch, {}).get(width)
        if entry is None:
            row += '{:>11}{:>8}'.format('-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>11.2f}{:>+8.2f}'.format(
            accuracy, accuracy - A_KL.get(width, accuracy))
    print(row)

print()
reference = sum(A_KL.values()) / len(A_KL)
print('{:22} mean {:.2f}   worst {:.2f}'.format(
    'A (reference)', reference, min(A_KL.values())))
for branch in BRANCHES:
    table = results.get(branch, {})
    if not table:
        continue
    accuracies = [100.0 * (1.0 - v[1]) for v in table.values()]
    mean = sum(accuracies) / len(accuracies)
    print('{:22} mean {:.2f}   worst {:.2f}   vs A {:+.2f}'.format(
        branch, mean, min(accuracies), mean - reference))

out = os.path.join(WORK, 'logs')
for branch in BRANCHES:
    log_dir = 'logs/cifar100_{}'.format(branch)
    if os.path.isdir(log_dir):
        shutil.copytree(log_dir, os.path.join(out, 'cifar100_' + branch),
                        dirs_exist_ok=True)
print('\ncheckpoints copied to', out)